# Chapter 12: Regression Testing with LLMs

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Smart Regression Suite** that uses LLMs to analyze code changes, identify impacted areas, prioritize regression tests, and generate new regression tests for modified features.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Understanding of regression testing strategy


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Change Impact Analysis

Analyze a code change or feature modification to determine which existing tests should be re-run.


In [ ]:
# Simulated change description (from a PR or change request)
change_description = {
    'change_id': 'CR-456',
    'title': 'Update payment processing to support Apple Pay',
    'description': 'Added Apple Pay as a payment method. Modified the payment gateway integration layer, updated the checkout UI to show Apple Pay button on Safari, and added new payment confirmation flow.',
    'files_changed': [
        'src/payments/gateway.py',
        'src/payments/apple_pay.py (new)',
        'src/checkout/payment_form.jsx',
        'src/checkout/confirmation.jsx',
        'src/api/payments_endpoint.py',
        'tests/test_payments.py'
    ],
    'modules_affected': ['payments', 'checkout', 'api']
}

# Existing test suite
existing_tests = [
    {'id': 'TC-001', 'name': 'Credit card payment success', 'module': 'payments', 'priority': 'Critical', 'last_run': 'Pass'},
    {'id': 'TC-002', 'name': 'Credit card declined', 'module': 'payments', 'priority': 'High', 'last_run': 'Pass'},
    {'id': 'TC-003', 'name': 'PayPal payment flow', 'module': 'payments', 'priority': 'High', 'last_run': 'Pass'},
    {'id': 'TC-010', 'name': 'Checkout page load', 'module': 'checkout', 'priority': 'Critical', 'last_run': 'Pass'},
    {'id': 'TC-011', 'name': 'Cart total calculation', 'module': 'checkout', 'priority': 'Critical', 'last_run': 'Pass'},
    {'id': 'TC-012', 'name': 'Shipping address validation', 'module': 'checkout', 'priority': 'High', 'last_run': 'Pass'},
    {'id': 'TC-020', 'name': 'Order confirmation email', 'module': 'notifications', 'priority': 'Medium', 'last_run': 'Pass'},
    {'id': 'TC-030', 'name': 'User login', 'module': 'auth', 'priority': 'Critical', 'last_run': 'Pass'},
    {'id': 'TC-031', 'name': 'User registration', 'module': 'auth', 'priority': 'High', 'last_run': 'Pass'},
    {'id': 'TC-040', 'name': 'Product search', 'module': 'search', 'priority': 'Medium', 'last_run': 'Pass'},
    {'id': 'TC-050', 'name': 'Refund processing', 'module': 'payments', 'priority': 'High', 'last_run': 'Fail'},
    {'id': 'TC-051', 'name': 'Payment API rate limiting', 'module': 'api', 'priority': 'Medium', 'last_run': 'Pass'},
]

# Analyze impact and select regression tests
impact_prompt = f"""Analyze this change and determine which existing tests need to be re-run.

Change: {json.dumps(change_description)}
Existing Tests: {json.dumps(existing_tests)}

For each test, determine:
- should_run: true/false
- risk_level: high/medium/low (how likely this change could break this test)
- reasoning: why it should or should not be included

Also identify:
- new_tests_needed: list of new test scenarios required for the Apple Pay feature
- integration_tests: cross-module tests that should be added

Return a JSON object with:
- regression_selection: array of {{test_id, should_run, risk_level, reasoning}}
- new_tests_needed: array of {{name, description, priority}}
- integration_tests: array of {{name, description, modules_involved}}
- estimated_execution_time: rough estimate in hours

Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': impact_prompt}],
    temperature=0.2
)
impact = json.loads(response.choices[0].message.content)

selected = [t for t in impact['regression_selection'] if t['should_run']]
print(f'Regression Suite: {len(selected)} of {len(existing_tests)} tests selected\n')
for t in impact['regression_selection']:
    marker = 'RUN' if t['should_run'] else 'SKIP'
    print(f"  [{marker}] {t['test_id']} - Risk: {t['risk_level']} - {t['reasoning'][:60]}")

## 2. Generate New Regression Tests

Create detailed regression test cases for the changed functionality.


In [ ]:
# Generate detailed regression tests for the new feature
new_tests = impact.get('new_tests_needed', [])
print(f'New tests needed: {len(new_tests)}\n')
for t in new_tests:
    print(f"  [{t['priority']}] {t['name']}")
    print(f"    {t['description'][:80]}")
    print()

# Generate full test cases for the top priority new tests
gen_prompt = f"""Generate detailed regression test cases for Apple Pay integration.

Required tests: {json.dumps(new_tests[:5])}
Change context: {json.dumps(change_description)}

For each test case provide:
- tc_id: TC-060+ format
- title: descriptive name
- preconditions: required setup
- steps: numbered list of detailed steps
- expected_results: what to verify at each step
- test_data: specific data values
- automation_feasibility: yes/no with brief note

Return as a JSON array. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': gen_prompt}],
    temperature=0.2
)
new_test_cases = json.loads(response.choices[0].message.content)
print(f'Generated {len(new_test_cases)} detailed test cases')
for tc in new_test_cases[:3]:
    print(f"\n{tc['tc_id']}: {tc['title']}")
    print(f"  Steps: {len(tc['steps'])}")
    print(f"  Automatable: {tc['automation_feasibility']}")

In [ ]:
# Build optimized regression test execution plan
plan_prompt = f"""Create an optimized regression test execution plan.

Selected regression tests: {json.dumps([t for t in impact['regression_selection'] if t['should_run']])}
New test cases: {json.dumps(new_test_cases)}

Optimize for:
1. Run highest-risk tests first
2. Group tests by module to minimize environment switches
3. Identify tests that can run in parallel
4. Estimate total execution time
5. Define pass/fail criteria for the regression suite

Return the plan as formatted text with clear phases and estimated times."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': plan_prompt}],
    temperature=0.3
)
print('REGRESSION TEST EXECUTION PLAN')
print('=' * 60)
print(response.choices[0].message.content)

## Exercises
1. Feed in a real git diff and have the LLM identify impacted test areas.
2. Build a test flakiness detector that analyzes test history to identify unreliable tests.
3. Create a test suite optimizer that removes redundant tests while maintaining coverage.
4. Generate automated test scripts (Selenium/Playwright) from the test case descriptions.
